In [2]:
import gem

# List all supported environments
# gem.print_envs()

# Initialize the environment
env = gem.make("game:FifteenPuzzle-v0-hard")

# Reset the environment to generate the first observation
observation, info = env.reset()
print(observation)

# Start the agent-environment loop
while True:
    action = env.sample_random_action()  # insert policy here, e.g.,
    # (pseudocode) action = llm.generate(observation)
    action = "\\boxed{right}"
    # apply action and receive next observation, reward
    # and whether the episode has ended
    next_observation, reward, terminated, truncated, info = env.step(action)
    print("ACT", action)
    print("OBS", next_observation)
    print("info:", info)
    


    # update the policy (online) here
    # e.g., policy = learn(policy, observation, action, reward, info)

    observation = next_observation
    # Exit when the episode terminates
    if terminated or truncated:
        break


You are playing the 15-Puzzle game.
You have to arrange the numbered tiles in ascending order from 1 to 15, with the empty space located in the bottom-right corner.
To make a move, you can slide a tile into the empty space (represented by a double underscore, e.g. __) by using one of the following commands:
- 'up': Move the tile below the empty space up.
- 'down': Move the tile above the empty space down.
- 'left': Move the tile to the right of the empty space left.
- 'right': Move the tile to the left of the empty space right.
To submit your move, type the direction (e.g., 'up', 'down', 'left', or 'right') in \boxed{...}.
The current board layout is shown below
Use logic and planning to solve the puzzle.

ACT \boxed{right}
OBS At turn 1, you made a valid move: right.

info: {'suffix': 'Here is the current board layout: \n 1  7 10 14\n 3 13  8  9\n11  4  5 12\n__  6  2 15\n\nEnter your move.'}
ACT \boxed{right}
OBS At turn 2, you chose a move right that is outside the bounds of the boa

In [ ]:
import os
import nltk
from builder import build_gem_envs, gem_projection


# Build a multi-env GEM vector (GuessTheNumber + Wordle)
multi_env = build_gem_envs(
    env_ids=["game:GuessTheNumber-v0-easy", "game:Wordle-v0-easy"],
    seed=0,
    env_num=2,   # two env workers, one for each sampled env id
    group_n=1,
    max_steps=5,
    resources_per_worker={"num_cpus": 0.1},
)

obs, infos = multi_env.reset()
print("initial env_ids:", [info.get("env_id") for info in infos])
print("obs count:", len(obs))

# Prepare one action per env (match env_num * group_n)
actions, valids = gem_projection([
    "\\boxed{5}",  # GuessTheNumber action
    "\\boxed{crane}",        # Wordle guess
], [info.get("env_id") for info in infos])
print("projected actions:", actions)
print("valid flags:", valids)

next_obs, rewards, dones, next_infos = multi_env.step(actions)
print("next_obs[0]:", next_obs[0])
print("next_obs[1]:", next_obs[1])
print("rewards:", rewards)
print("dones:", dones)
print("next env_ids:", [info.get("env_id") for info in next_infos])

multi_env.close()

/projectnb/replearn/xfl/.conda/envs/verl-agent/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-12 17:44:02,110	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2025-12-12 17:44:04,984	INFO worker.py:1942 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
(GemWorker pid=109196) [nltk_data] Downloading package words to
(GemWorker pid=109196) [nltk_data]     /usr3/graduate/xfl/nltk_data...
(GemWorker pid=109196) [nltk_data]   Package words is already up-to-date!
(GemWorker pid=109196) [nltk_data] Downloading package words to
(GemWorker pid=109196) [nltk_data]     /usr3/graduate/xfl/nltk_data...
(GemWorker pid=109196) [nltk_data]   Package words is already up-to-date!


initial env_ids: ['game:Wordle-v0-easy', 'game:Wordle-v0-easy']
obs count: 2
projected actions: ['\\boxed{5}', '\\boxed{crane}']
valid flags: [True, True]
next_obs[0]: At turn 1, you did not provide a valid guess.
next_obs[1]: At turn 1, you guessed CRANE which has 5 letters but the secret word has 3 letters.
rewards: [-0.1, 0.0]
dones: [True, False]
next env_ids: ['game:Wordle-v0-easy', 'game:Wordle-v0-easy']


(GemWorker pid=109198) [nltk_data] Downloading package words to
(GemWorker pid=109198) [nltk_data]     /usr3/graduate/xfl/nltk_data...
(GemWorker pid=109198) [nltk_data]   Package words is already up-to-date!


In [ ]:
from builder import (
    build_gem_envs,
    gem_projection,
    GEM_TASK_POOL_TRAIN,
)

# Build with default train pool (fixed tasks + seeds)
multi_env = build_gem_envs(
    env_ids=[t["env_id"] for t in GEM_TASK_POOL_TRAIN],
    seed=0,
    env_num=2,
    group_n=1,
    max_steps=5,
    resources_per_worker={"num_cpus": 0.1},
    use_default_pool=True,
    is_train=True,
)

# First reset samples tasks from train pool
obs, infos = multi_env.reset()
print("reset env_ids:", [info.get("env_id") for info in infos])
print("obs count:", len(obs))

# Prepare actions (simple placeholders)
actions, valids = gem_projection([
    "\\boxed{5}",  # GuessTheNumber-compatible
    "crane",        # Wordle-style text or other
], [info.get("env_id") for info in infos])
next_obs, rewards, dones, next_infos = multi_env.step(actions)
print("rewards:", rewards)
print("dones:", dones)
print("next env_ids:", [info.get("env_id") for info in next_infos])

# Soft reset the first worker to the same task instance
obs_map, info_map = multi_env.soft_reset([0])
print("soft_reset[0] env_id:", info_map[0].get("env_id"))

# Hard reset again (may sample a new task)
obs2, infos2 = multi_env.reset()
print("post-reset env_ids:", [info.get("env_id") for info in infos2])

multi_env.close()

reset env_ids: ['game:Sudoku-v0-easy', 'game:Hangman-v0-easy']
obs count: 2
rewards: [-0.1, -0.1]
dones: [True, True]
next env_ids: ['game:Sudoku-v0-easy', 'game:Hangman-v0-easy']
soft_reset[0] env_id: game:Sudoku-v0-easy
post-reset env_ids: ['game:Mastermind-v0-easy', 'game:GuessTheNumber-v0-easy']


(GemWorker pid=109204) [nltk_data] Downloading package words to
(GemWorker pid=109204) [nltk_data]     /usr3/graduate/xfl/nltk_data...
(GemWorker pid=109204) [nltk_data]   Package words is already up-to-date!


In [8]:


# Fixed task pool with a single GuessTheNumber instance (seeded)
TASK_POOL_GTN = [{"env_id": "game:GuessTheNumber-v0-easy", "seed": 123}]

# Build env with one worker and deterministic task
env = build_gem_envs(
    env_ids=["game:GuessTheNumber-v0-easy"],
    seed=0,
    env_num=1,
    group_n=1,
    max_steps=10,
    resources_per_worker={"num_cpus": 0.1},
    task_pool=TASK_POOL_GTN,
    is_train=True,
)

# Helper to roll a fixed action sequence and capture observations
def roll_sequence(actions):
    obs_list = []
    infos_list = []
    rewards_list = []
    dones_list = []
    obs, infos = env.reset()
    obs_list.append(obs[0])
    infos_list.append(infos[0])
    for a in actions:
        acts, valids = gem_projection([a], [infos_list[-1].get("env_id")])
        nxt_obs, rwds, dns, nxt_infos = env.step(acts)
        obs_list.append(nxt_obs[0])
        infos_list.append(nxt_infos[0])
        rewards_list.append(rwds[0])
        dones_list.append(dns[0])
    return obs_list, infos_list, rewards_list, dones_list

# Fixed guess sequence
GUESS_SEQ = ["\\boxed{2}", "\\boxed{1}", "\\boxed{3}", "\\boxed{4}"]

# Roll before soft reset
obs_a, infos_a, rewards_a, dones_a = roll_sequence(GUESS_SEQ)

# Soft reset the same worker and roll the identical sequence
obs_map, info_map = env.soft_reset([0])
obs_b, infos_b, rewards_b, dones_b = roll_sequence(GUESS_SEQ)

print("--- Pre-soft-reset observations ---")
for i, o in enumerate(obs_a):
    print(f"step {i}: {o}")

print("--- Post-soft-reset observations ---")
for i, o in enumerate(obs_b):
    print(f"step {i}: {o}")

env.close()

--- Pre-soft-reset observations ---
step 0: You are playing Guess The Number.
You have to guess the number between 1 and 10 (inclusive) within 4 turns.
As you enter your guess, the game will provide you with hints such as the target number is 'higher' or 'lower'.
You may provide your response in any manner. Only the number that is wrapped inside \boxed{} will be considered as your guess, for example, \boxed{5}.
As you play, the history of your guesses will be appended below. Use the information to complete the game before you run out of guesses.
Enter your first guess to start the game.

step 1: At turn 1, you guessed 2, and the target number is lower than 2.
step 2: Congratulations! You guessed the correct number 1 in 2 turns.
step 3: At turn 3, you guessed 3, and the target number is lower than 3.
step 4: You have reached the maximum number of turns.
--- Post-soft-reset observations ---
step 0: You are playing Guess The Number.
You have to guess the number between 1 and 10 (inclusive

In [6]:
print(dones_a)
print(rewards_a)


[False, True, False, True]
[0.0, 1.0, 0.0, 0.6666666666666667]
